# 03 Using Tools
In this workbook we'll discuss how to integrate tools with your models.

In [21]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [22]:
from langchain.tools import tool

# you annotate a tool with the @tool decorator
# you MUST define a doc comment for the function


@tool
def square_root(num: float) -> float:
    """returns the square root of a number

    Args:
        num (float): number whose square root is desired
    Returns:
        (float): the square root of the number
    """
    print(f"------ Calling square_root({num}) tool ------")
    return num**0.5

In [23]:
# you can invoke a too directly - note how params aee passed
response = square_root.invoke({"num": 4156})
print(response)

------ Calling square_root(4156.0) tool ------
64.4670458451448


In [24]:
# you can add additional info to the tool decorator


@tool("square_root", description="Calculate the square root of a number")
def tool1(num: float) -> float:
    """returns the square root of a number

    Args:
        num (float): number whose square root is desired
    Returns:
        (float): the square root of the number
    """
    return num**0.5

In [25]:
response = tool1.invoke({"num": 4156})
print(response)

64.4670458451448


### Adding tools to agent
Here is how you can add tools to an agent

In [26]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt="""You are a math wizard that can preform calculations 
        using tools provided to you. You have the following tools:
        - square_root - to calculate square root of number.
        Always use tools first before relying on your own ability""",
    tools=[square_root],
)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Calculate square root of 4156",
            }
        ]
    }
)
print(response["messages"][-1].content)

------ Calling square_root(4156.0) tool ------
The square root of 4156 is approximately 64.4670458451448. 
(About 64.4670 when rounded to four decimal places.)


Notice that the response is not structured. I would like to get just the funal result - 64.4670458451448, here is where I'll used structured output

In [27]:
from pydantic import BaseModel


class MathResult(BaseModel):
    result: float


agent2 = create_agent(
    model="openai:gpt-5-nano",
    system_prompt="""You are a math wizard that can preform calculations 
        using tools provided to you. You have the following tools:
        - square_root - to calculate square root of number.
        Always use tools first before relying on your own ability""",
    tools=[square_root],
    response_format=MathResult,
)

response = agent2.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Calculate square root of 4156",
            }
        ]
    }
)
print(f"Result -> {response["structured_response"].result:.3f}")

------ Calling square_root(4156.0) tool ------
Result -> 64.467


### Add a web search tool using Tavily Search
In this section, we'll see a specific case of using web-search with an agent. Specifically, we'll use Tavily search.

In [28]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""

    return tavily_client.search(query)


web_search.invoke("Who is the current mayor of San Francisco?")

ModuleNotFoundError: No module named 'tavily'